# **HydroServer Exercise 2: Loading Sensor Data using the Streaming Data Loader**


### **Overview**

This exercise creates a datastream in the workspace you set up in Exercise 1. Later, you will use this datastream to load observations with the [HydroServer Streaming Data Loader](https://hydroserver.org/user-guides/tutorials/hydroserver-101/part-3-sdl-setup.html).

### **Prerequisites**

To complete this exercise, you must first complete Exercise 1, where you create the workspace that will contain your datastream.

### **Software Requirements**

This notebook was developed using Python Verion 3.14 and Version 1.11 of the hydroserverpy Python package.

## 1. **Getting Started**

---

### **Import Required Packages**

For this example, we'll use hydroserverpy, pandas, and the datetime packages.

In [ ]:
!pip install hydroserverpy==1.11
from hydroserverpy import HydroServer
import pandas as pd
from datetime import datetime
from getpass import getpass

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 114.8/114.8 kB 2.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 46.7/46.7 kB 4.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 331.1/331.1 kB 8.0 MB/s eta 0:00:00


## 1. **Connect to HydroServer**

---

Initialize the connection to HydroServer as we did in Exercise 1.

In [ ]:
# Set initial parameters to connect to HydroServer
hydroserver_host = 'https://playground.hydroserver.org'

# Change the email and password below to your HydroServer username and password
hydroserver_email = 'svicario@lincolninst.edu' #'user@youremail.com'
hydroserver_password = getpass('Enter your HydroServer password: ') #getpass('Enter your HydroServer password: ')

# Initialize HydroServer connection with credentials.
hs = HydroServer(
    host=hydroserver_host,
    email=hydroserver_email,
    password=hydroserver_password
)

print('\nSuccessfully connected to HydroServer!')

Enter your HydroServer password: ··········

Successfully connected to HydroServer!


Get the ID of your workspace

In [ ]:
workspaces = hs.workspaces.list(
    is_associated=True
).fetch_all()

for workspace in workspaces.items:
    print("Workspace name:", workspace.name)
    print("Workspace ID:", workspace.uid)
    print()

workspace_id = hs.workspaces.list(is_associated=True).items[0].uid

Workspace name: Rwanda Training 2026
Workspace ID: 019ffae0-8a1d-7c6f-add4-f52cdcc3a2df



## 2. **Set Up a New Datastream for Loading Real-time Data**

---

This is a follow-up to Exercise 1. In the previous exercise, we created a workspace and a monitoring station. Now, we only need to create a new datastream at the same location to store the current observations collected by the telemetry station at the Kanzenze monitoring site.

The data were downloaded from the [Rwanda Water Portal Kanzenze Station](https://waterportal.rwb.rw/index.php/location_ng_info/259501?).

HydroServer uses a modified version of the OGC SensorThings API data model for storing time series data and their associated metadata. HydroServer's data model includes the following important entities that we need to create before loading data:

* **Thing**: A monitoring station on which or at which data were collected (e.g., a streamflow gage or weather station).
* **Observed Property**: The variable that is measured (e.g., discharge, water temperature, etc.).
* **Units of Measure**: The units of measure associated with the Observation values (e.g, cubic meters per second).
* **Sensor**: The instrument or method used to measure or create the Observation values.
* **Processing Level**: The degree of processing that has been applied to the Observation values (e.g., "Raw" or "Quality Controlled").
* **Datastream**: A description of the time series that includes all of these attributes.

Once all of these metadata tables have been populated, the time series of data values can be loaded to the **Observations** table in the database.

**NOTE**: To create objects in HydroServer, you will have to pass their required and optional metadata elements. For more information about HydroServer's data model and a data dictionary that describes each of the entities, see HydroServer's documentation at https://www.hydroserver.org.

### **Create a Sensor**

The OGC SensorThings API data model refers to the method used for creating observations as the "Sensor". In many cases this will be a physical sensor installed at the monitoring site. But, sometimes other methods are used to create observations. We need to create the metadata describing this so potential data users know how the data were created.

**NOTE**: The specific metadata required when creating metadata for a Sensor is dependent upon the "Method Type". For instrument deployments, specific information about the manufacturer and model of the sensor should be specified. For "Derivation" methods, the name and description are required, and a method_code and method_link can be specified if needed.

In [ ]:
real_time_stage_sensor = hs.sensors.create(
    workspace=workspace_id,
    name='Kanzenze Real-Time Stage Observations',
    description='Real-Time stage observations recorded at the Kanzenze hydrological station.',
    encoding_type='application/json',
    method_type='Observation',
    method_code='kanzenze-real-time-stage'
)

### **Create an Observed Property**

Observed Properties are the variables measured at the monitoring site. Similar to creating the monitoring site (Thing), we need to know the required and optional metadata elements for Observed Properties so we can pass them to the ```create()``` method. For this example, we will load one time series from a CSV file

In [ ]:
stage = hs.observedproperties.create(
    workspace=workspace_id,
    name='Stage',
    definition='Stage',
    description='Stage is the height of the water surface at a monitoring location relative to a reference level.',
    observed_property_type='Hydrology',
    code='Stage'
)

print("Created observed property:")
print(f"{stage.name}: {stage.uid}")

Created observed property:
Stage: 019ffb16-2e1d-76a5-85ae-38a8d7324054


### **Create Units of Measure**

Next we need to add metadata to specify the Units of measure used for recording the data in the CSV file. We need one unit for each of the columns of data we are loading because they are all different.

In [ ]:
stage_unit = hs.units.create(
    workspace=workspace_id,
    name='Meter',
    symbol='m',
    definition='Unit for water stage',
    unit_type='Length'
)

print("Created unit:")
print(f"{stage_unit.name}: {stage_unit.uid}")

Created unit:
Meter: 019ffb16-3420-73ae-a68e-4b0bcc148501


### **Create a Processing Level**

In HydroServer, the Processing Level indicates the degree of processing a datastream has been subject to. For example, data can be "Raw", which means that they were recorded in the field and nobody has looked at them yet, or they could be "Quality Controlled", which means that a technician has reviewed the data. All of the data we are loading right now are raw observations from the field with no processing, so we need a Processing Level that indicates this.

In [ ]:
new_processing_level = hs.processinglevels.create(
    workspace=workspace_id,
    code='Raw',
    definition='Raw Data',
    explanation='Data that have not been processed or quality controlled.'
)

print("Created processing levels:")
print(f"{new_processing_level.code}: {new_processing_level.uid}")

Created processing levels:
Raw: 019ffb16-397d-73ac-ab57-1c80ad1616fb


### **Create a Datastream**

The last step before loading data is to create metadata for each of the "Datastreams". This helps us link the time series values to where they were measured, which Observed Property they represent, which Units they are recorded in, etc. In the following code, we create the necessary datastream metadata, using the UUIDs for the other metadata entities we created above for the three datastreams we want to load data to.

**NOTE**: Since these Datastreams are new, they don't contain any Observation values yet. We'll set the ```value_count=0``` and arbitrarily set the ```phenomenon_begin_time``` and ```phenomenon_end_time```. Those will get reset when we load Observation values. All of the LRO aquatic sensor data have spacing of 15-minutes, so we set that accordingly. Each Datastream has a name and description that we'll set using some attributes of the Datastream, but you can name these accoding to your own naming conventions.

In [ ]:
ds_stage = hs.datastreams.create(
    name=f"{stage.name} - Real-time - {new_thing.name}",
    description=f'Real-time {stage.name.lower()} observations at {new_thing.name}',
    thing=new_thing.uid,
    sensor=real_time_stage_sensor.uid,
    observed_property=stage.uid,
    processing_level=new_processing_level.uid,
    unit=stage_unit.uid,
    observation_type='Field Observation',
    result_type='Timeseries',
    sampled_medium='Surface Water',
    no_data_value=-9999,
    aggregation_statistic='Continuous',
    time_aggregation_interval=0,
    time_aggregation_interval_unit='minutes',
    intended_time_spacing=1,
    intended_time_spacing_unit='days',
    status='Complete',
    value_count=0,
    phenomenon_begin_time=datetime(year=2022, month=9, day=28),
    is_private=False,
    is_visible=True
)

print("Created datastream:")
print(f"{ds_stage.name}: {ds_stage.uid}")

Created datastream:
Stage - Real-time - Kanzenze Hydrological Station: 019ffb1a-0b79-72b4-967b-27e2af807354
